# Coding Challenge: Leverage (Bitcoin)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:,.6f}'.format
plt.style.use('default')


## 1) Calculate Levered Returns for Bitcoin (Leverage = 4)

We use daily **simple returns** and apply 4x leverage:

`levered_return = leverage * simple_return`

To keep simulation realistic for a retail-style account, we cap daily return at -100% (account cannot go below zero in a one-day mark-to-market model).

In [2]:
prices = pd.read_csv('close.csv', index_col='Date', parse_dates=['Date']).sort_index()

btc_col_candidates = ['BTC-USD', 'BTCUSD', 'BTC_USD']
btc_col = next((col for col in btc_col_candidates if col in prices.columns), None)
if btc_col is None:
    raise ValueError('Bitcoin column not found in close.csv (expected BTC-USD).')

btc = prices[[btc_col]].dropna().rename(columns={btc_col: 'Close'})
btc['simple_return'] = btc['Close'].pct_change()
btc = btc.dropna(subset=['simple_return']).copy()

leverage = 4
btc['levered_return_raw'] = leverage * btc['simple_return']
btc['levered_return'] = np.where(btc['levered_return_raw'] < -1, -1, btc['levered_return_raw'])

btc[['simple_return', 'levered_return']].head()

,simple_return,levered_return
Date,,
2014-10-02,-0.022270,-0.089079
2014-10-03,-0.041485,-0.165941
2014-10-04,-0.085243,-0.340973
2014-10-05,-0.025408,-0.101634
2014-10-06,0.029856,0.119422


## 2) Visualize and Compare with Unlevered Investment

We compare the growth of $1 invested:
- Unlevered Bitcoin
- 4x Levered Bitcoin (with -100% daily cap)

In [4]:
btc['growth_unlevered'] = (1 + btc['simple_return']).cumprod()
btc['growth_levered_4x'] = (1 + btc['levered_return']).cumprod()

print('Final growth multiples:')
print(f"Unlevered: {btc['growth_unlevered'].iloc[-1]:.6f}x")
print(f"Levered 4x: {btc['growth_levered_4x'].iloc[-1]:.6f}x")

Final growth multiples:
Unlevered: 93.005044x
Levered 4x: 0.000000x


### Interpreting the Growth Result

If you see something like:
- `Unlevered: 93.005044x`
- `Levered 4x: 0.000000x`

it means path-risk overwhelmed leverage.

- The unlevered path benefited from Bitcoin's long-run upside, so $1 compounded strongly.
- The 4x levered path encountered at least one day where the leveraged loss reached or exceeded -100% (after scaling and capping), which represents margin wipeout/forced closeout in this simplified model.
- Once cumulative value hits zero, it cannot recover because future returns are applied to zero capital.

Key lesson: leverage magnifies every daily move, so a single extreme negative day can dominate long-term outcome even when the underlying asset performs very well over the full period.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(btc.index, btc['growth_unlevered'], label='Unlevered BTC')
plt.plot(btc.index, btc['growth_levered_4x'], label='Levered BTC (4x)')
plt.title('Bitcoin: Unlevered vs 4x Levered Growth')
plt.xlabel('Date')
plt.ylabel('Growth of $1')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
daily_stats = pd.DataFrame({
    'Series': ['Unlevered BTC', 'Levered BTC (4x)'],
    'Mean Daily Return': [btc['simple_return'].mean(), btc['levered_return'].mean()],
    'Std Daily Return': [btc['simple_return'].std(ddof=1), btc['levered_return'].std(ddof=1)],
    'Worst Daily Return': [btc['simple_return'].min(), btc['levered_return'].min()],
    'Best Daily Return': [btc['simple_return'].max(), btc['levered_return'].max()]
})

daily_stats

## 3) Is Extreme Leverage (>100x) a Good Idea Without Advanced Risk Management?

Short answer: **No**.

Reasoning:
- Bitcoin is highly volatile; even small adverse moves can wipe out heavily levered positions.
- At 100x leverage, about a 1% adverse move can fully consume margin in a simplified mark-to-market view.
- Liquidation risk becomes extremely high, and compounding works against you after large losses.
- Without advanced tools (strict position sizing, dynamic hedging, stop systems, latency-aware execution, and exchange-specific liquidation modeling), survival probability is low.

Practical conclusion: very high leverage is generally a poor idea without professional-grade risk controls.